# Model Serving with a REST API

In production, a model does not sit in a Jupyter notebook waiting for you to call `model.predict()`. Other services—mobile apps, dashboards, backend jobs—need to reach it over HTTP. This notebook wraps a FastAPI service around **the model you trained earlier in the diploma**, writes the service to disk, and then calls it from Python.

You do not train anything here. Unit 1 of a deployment course is where you stop being the person who trains models and start being the person who ships them.

**Learning objectives**
1. Explain why HTTP is the standard interface for deployed models.
2. Load your **portfolio model** (from AIAT 114 or AIAT 122) plus its model card, and read the card as the API contract.
3. Generate a Pydantic request schema *from the card* so the API cannot drift away from the model.
4. Implement `/predict` and `/health` endpoints that load the artifact once, at process start.
5. Test the real endpoint in-process, without running a live server.

> **First time here?** Read [`../../PORTFOLIO_MODEL.md`](../../PORTFOLIO_MODEL.md) — it tells you how to export a model from AIAT 114 or AIAT 122 into the portfolio directory this course reads. If you have not exported one yet, the notebooks build a **named fallback** (`wdbc-baseline`) so nothing blocks you, and they say so in their output.

## 🔗 Where this fits

**Builds on:** Course 08 (AIAT 122) — Unit 5, lesson 06 "06 Flask and FastAPI Deployment" — you already served a model behind a single FastAPI endpoint there; this course starts from that endpoint and adds what a real service needs: a typed request schema, a health check, and loading the model once at startup rather than per request.

**Used later in:** Course 12 (AIAT 126) — Unit 1, lesson 01 "Project Proposal and Literature Review", whose worked proposal lists "deploy model as a web application" among the project objectives and web-based deployment in scope.


## 📰 The feature file that took Cloudflare down for six hours

On **18 November 2025** a routine database-permissions change at Cloudflare made an internal query start returning duplicate rows. That query builds the *feature file* — the list of inputs to the Bot Management machine-learning model — and the duplicates roughly doubled its size. The proxy that reads the file had a hard limit of **200 machine-learning features**; the oversized file crossed it and the request-handling process panicked instead of rejecting the file cleanly. The file is regenerated and pushed to every machine on the network **every five minutes**, so the fault spread as fast as the update did. Cloudflare's own post-mortem puts the disruption from **11:20 to 17:06 UTC** — about six hours of HTTP 5xx across a large slice of the internet, caused not by a bad model but by a broken *contract* between a model and the code serving it.

**What goes wrong without this lesson.** `model.predict()` in a notebook has no contract at all — you are the only caller and you remember the column order. The moment another team reaches your model over HTTP, three facts have to live somewhere a machine can check: *which* features, in *which* order, of *which* type. When they do not, the failure is never a helpful error message. It is a confident wrong answer, or a crash in the very layer that was supposed to protect you. Every design choice below exists to make that contract explicit and machine-checkable: the card carries it, Pydantic enforces it, and `/health` tells you which model is really behind the URL.

## 1  Why HTTP? The problem with `model.predict()` in production

A trained model lives as a Python object in RAM. Other teams write in JavaScript, Go, or Java; mobile apps run on iOS. None of them can import your Python object. An HTTP API solves this: any language can send a JSON request and receive a JSON response. The model stays in one place; everyone else calls it over the network.


## 1.1  HTTP and REST in five minutes (primer)

This course talks to models over the web, so make sure these terms are solid before writing any code. (You have met this before — twice. AIAT 115 Unit 5 `08_deployment.ipynb` pickled a model, wrote a `model_metadata.json` beside it, and served it from both Flask and FastAPI; AIAT 122 Unit 5 `06_flask_fastapi_deployment.ipynb` tested a `POST /predict` endpoint with `TestClient`. Both were short first looks that served a stand-in predictor. This primer fixes the vocabulary properly, and from here on every endpoint you build serves a real trained artifact — yours.)

- **HTTP** is the request/response protocol of the web. A **client** (browser, mobile app, another service) sends a *request* to a **server**; the server sends back a *response*. Every request names a **method** and a **URL path**:
  - `GET /health` — "read something" (no body needed)
  - `POST /predict` — "send data, get a result" (the data travels in the request **body**, usually as **JSON** like `{"mean_radius": 12.4, ...}`)
- Every response carries a **status code**: `200` = success, `404` = no such path, `422` = your JSON was malformed, `500` = the server crashed. Our API will use these deliberately.
- A **REST API** is simply an HTTP server organised around such method + path pairs (called **endpoints**). "Calling the model" becomes "POSTing JSON to `/predict`".
- `curl` is the command-line tool for sending test requests: `curl -X POST http://localhost:8000/predict -d '{...}'`.

That is the whole mental model: *JSON in over HTTP, JSON out, status code tells you what happened.* Everything else in this notebook builds on it.

## 2  Install dependencies and load your portfolio model

In [1]:
# Run this once; restart kernel if packages were newly installed
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn[standard]", "scikit-learn", "joblib", "pydantic", "httpx"],
               check=False)
print("Dependencies ready")


Dependencies ready



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# WHAT: put the Course 11 root on the import path, then load YOUR portfolio model.
# WHY: deployment never starts with training - it starts with an artifact someone
# else finished. In this diploma that someone is you, in AIAT 114 (Course 04) or
# AIAT 122 (Course 08). Course 11/PORTFOLIO_MODEL.md shows how to export it.
import json
import sys
from pathlib import Path

# Walk up the folder tree to find the Course 11 root, where portfolio_model.py lives.
for _d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_d / "portfolio_model.py").exists():
        COURSE11 = _d
        break
else:
    raise FileNotFoundError("Could not find portfolio_model.py - run this notebook from inside Course 11.")

if str(COURSE11) not in sys.path:
    sys.path.insert(0, str(COURSE11))

import portfolio_model as pf

# No portfolio model exported yet? This builds the NAMED FALLBACK 'wdbc-baseline'
# (the real Wisconsin breast-cancer study that ships inside scikit-learn, so it
# needs no download) and says so, loudly, in the output below.
model, card = pf.load_portfolio_model()
MODEL_DIR = pf.portfolio_dir()


No portfolio model found at /Users/abdullah/ai-diploma-portfolio
-> Building the NAMED FALLBACK 'wdbc-baseline' so this lesson can run.


FALLBACK MODEL 'wdbc-baseline' — this is NOT your model.
  directory   : /Users/abdullah/ai-diploma-portfolio
  artifact    : model.joblib  (sklearn)
  task        : classification  ->  2 classes ['malignant', 'benign']
  features    : 30 (first three: ['mean radius', 'mean texture', 'mean perimeter'])
  accuracy    : 0.9825 on held-out 20% (random_state=42, stratified)
  Export your own model from AIAT 114 or AIAT 122 and re-run: see Course 11/PORTFOLIO_MODEL.md


## 3  The artifact you are about to serve

The portfolio directory holds exactly two files:

```
~/ai-diploma-portfolio/
├── model.joblib       ← the trained artifact (or model.onnx / model_scripted.pt)
└── model_card.json    ← the contract: feature order, class names, sample input, score
```

> **Why a card and not just the model file?** `joblib.load(path)` gives you back an object that can `.predict()`, and nothing else. It will not tell you that column 3 must be *mean area* and not *mean perimeter* — it will happily predict from a scrambled row and return a confident, wrong answer. The card is the machine-readable answer to "what does this thing expect, and what do its outputs mean?". Everything the API needs beyond the weights comes from it.

Serialization formats themselves (pickle vs joblib vs ONNX) are covered in notebook **02_model_packaging**; here we only need "load the artifact, read the card".

In [3]:
# WHAT: read the model card - the contract that travels next to the artifact.
# WHY: a server cannot guess three things from a model file: the ORDER of the
# input features, what the class indices mean, and one real row to smoke-test with.
# The card carries all three, which is why every lesson from here reads it first.
print("Model            :", card["name"], "  from", card["source_course"])
print("Framework        :", card["framework"], "->", MODEL_DIR / card["artifact"])
print("Feature count    :", len(card["feature_names"]))
print("First 5 features :", card["feature_names"][:5])
print("Class names      :", card["class_names"])
print(f"Reported score   : {card['metric']['name']} = {card['metric']['value']} "
      f"({card['metric']['split']})")

# Prove the artifact really predicts before we wrap HTTP around it.
# card["sample_input"] is one real feature row saved at export time.
proba = model.predict_proba([card["sample_input"]])[0]
best = int(proba.argmax())
print("\nSmoke test on card['sample_input']:")
print(f"  predicted class : {card['class_names'][best]}")
print(f"  confidence      : {proba[best]:.1%}")


Model            : wdbc-baseline   from AIAT 125 fallback
Framework        : sklearn -> /Users/abdullah/ai-diploma-portfolio/model.joblib
Feature count    : 30
First 5 features : ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness']
Class names      : ['malignant', 'benign']
Reported score   : accuracy = 0.9825 (held-out 20% (random_state=42, stratified))

Smoke test on card['sample_input']:
  predicted class : benign
  confidence      : 99.9%


## 4  Define the FastAPI application

Three key components:
- **Schema built from the card** — one required `float` field per entry in `feature_names`, generated at import time by `pydantic.create_model`. Change the model, re-export the card, and the schema follows.
- **Load at import, not per request** — the module-level `joblib.load` runs once per server process. Loading inside `/predict` would pay that cost on every single call.
- **`/predict` and `/health`** — predict returns the class name and confidence; health lets load balancers check the service is alive *and* reports which model is live (including whether it is the fallback).

In [4]:
%%writefile /tmp/main.py
# WHAT: write the FastAPI server to /tmp/main.py (this cell only creates the file).
# WHY: the app reads model_card.json and builds its request schema FROM the card,
# so the API contract can never drift away from the model it serves. Hard-coding
# field names is exactly how a served API ends up lying about its own model.
"""FastAPI service for the AIAT 125 portfolio model."""
import json
import os
import re
from pathlib import Path

import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, Field, create_model

# --- 1. Find the artifact ---------------------------------------------------
# A deployed app never imports course helpers; it depends only on the artifact
# and its card. Same two files, whether they sit in your home directory or in
# a container image.
MODEL_DIR = Path(os.environ.get("AI_DIPLOMA_PORTFOLIO",
                                str(Path.home() / "ai-diploma-portfolio")))
CARD = json.loads((MODEL_DIR / "model_card.json").read_text())
ARTIFACT = MODEL_DIR / CARD["artifact"]

# --- 2. Load it ONCE, at import time (= once per server process) ------------
if CARD["framework"] == "sklearn":
    import joblib
    _model = joblib.load(ARTIFACT)

    def probabilities(rows):
        return np.asarray(_model.predict_proba(rows))

elif CARD["framework"] == "onnx":
    import onnxruntime as ort
    _session = ort.InferenceSession(str(ARTIFACT), providers=["CPUExecutionProvider"])
    _input = _session.get_inputs()[0].name

    def probabilities(rows):
        logits = np.asarray(_session.run(None, {_input: np.asarray(rows, dtype=np.float32)})[0])
        exp = np.exp(logits - logits.max(axis=1, keepdims=True))
        return exp / exp.sum(axis=1, keepdims=True)

else:
    raise RuntimeError(
        f"This app serves 'sklearn' or 'onnx' artifacts, not {CARD['framework']!r}."
    )

# --- 3. Build the request schema FROM the card ------------------------------
def api_field(name):
    """Turn a training column name ('mean radius') into a JSON field ('mean_radius')."""
    slug = re.sub(r"\W+", "_", str(name).strip().lower()).strip("_")
    return f"f_{slug}" if not slug or slug[0].isdigit() else slug

FIELD_ORDER = [api_field(n) for n in CARD["feature_names"]]

# create_model builds a Pydantic class at runtime - one required float per feature.
PredictRequest = create_model(
    "PredictRequest",
    **{field: (float, Field(..., description=original))
       for field, original in zip(FIELD_ORDER, CARD["feature_names"])},
)

class PredictResponse(BaseModel):
    prediction: str
    class_id: int
    confidence: float

# --- 4. Endpoints -----------------------------------------------------------
app = FastAPI(title=f"{CARD['name']} API", version="1.0")

@app.get("/health")
def health():
    """Load balancers poll this; it also tells you WHICH model is live."""
    return {"status": "ok",
            "model": CARD["name"],
            "source": CARD["source_course"],
            "is_fallback": bool(CARD.get("is_fallback", False))}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    # Rebuild the row in the EXACT order the model was trained on. Getting this
    # order wrong is silent: the model still returns a number, just a wrong one.
    row = [getattr(req, field) for field in FIELD_ORDER]
    proba = probabilities([row])[0]
    idx = int(np.argmax(proba))
    return PredictResponse(prediction=CARD["class_names"][idx],
                           class_id=idx,
                           confidence=round(float(proba[idx]), 4))


Overwriting /tmp/main.py


The file was just written to `/tmp/main.py`. To run the real server:

```bash
uvicorn main:app --host 0.0.0.0 --port 8000
```

The request body has one field per entry in your card's `feature_names`, so it is different for every student's model. The next cell prints a ready-to-paste `curl` for *your* model rather than guessing at one here.

## 5  Call the real endpoint, in-process

We cannot start a live uvicorn server inside a notebook, but we can import `/tmp/main.py` and call its `predict` handler directly. That is the same function uvicorn would call — same schema, same artifact, same code path — so it is a real test, not a re-implementation.

In [5]:
# WHAT: import the app module and call the real /predict handler in-process.
# WHY: this exercises the code we would deploy - schema validation included -
# without a uvicorn process, so a failure here is a failure in production code.
import importlib
import sys

sys.path.insert(0, "/tmp")
import main as serving_app
importlib.reload(serving_app)

# The card's sample_input is a real feature row; zip it onto the API field names.
payload = dict(zip(serving_app.FIELD_ORDER, card["sample_input"]))
response = serving_app.predict(serving_app.PredictRequest(**payload))

print("Serving      :", serving_app.CARD["name"], f"({len(payload)} input fields)")
print("Health       :", serving_app.health())
print("Prediction   :", response.prediction, f"(class_id={response.class_id})")
print(f"Confidence   : {response.confidence:.1%}")

# A ready-to-paste curl for YOUR model, built from the same card.
print("\ncurl for this exact request:")
print("curl -X POST http://localhost:8000/predict \\")
print("     -H 'Content-Type: application/json' \\")
print(f"     -d '{json.dumps(payload)}'")


Serving      : wdbc-baseline (30 input fields)
Health       : {'status': 'ok', 'model': 'wdbc-baseline', 'source': 'AIAT 125 fallback', 'is_fallback': True}
Prediction   : benign (class_id=1)
Confidence   : 99.9%

curl for this exact request:
curl -X POST http://localhost:8000/predict \
     -H 'Content-Type: application/json' \
     -d '{"mean_radius": 11.26, "mean_texture": 19.96, "mean_perimeter": 73.72, "mean_area": 394.1, "mean_smoothness": 0.0802, "mean_compactness": 0.1181, "mean_concavity": 0.09274, "mean_concave_points": 0.05588, "mean_symmetry": 0.2595, "mean_fractal_dimension": 0.06233, "radius_error": 0.4866, "texture_error": 1.905, "perimeter_error": 2.877, "area_error": 34.68, "smoothness_error": 0.01574, "compactness_error": 0.08262, "concavity_error": 0.08099, "concave_points_error": 0.03487, "symmetry_error": 0.03418, "fractal_dimension_error": 0.006517, "worst_radius": 11.86, "worst_texture": 22.33, "worst_perimeter": 78.27, "worst_area": 437.6, "worst_smoothness": 0.

## 6  What happens when the schema is violated

Pydantic enforces the schema before the model ever runs. Below we show what validation errors look like.


In [6]:
# WHAT: feed the card-generated schema deliberately broken requests.
# WHY: an API must reject bad input with a clear 422 instead of handing garbage
# to the model - and note we never wrote a single validation rule ourselves.
from pydantic import ValidationError

PredictRequest = serving_app.PredictRequest
complete = dict(zip(serving_app.FIELD_ORDER, card["sample_input"]))
first, second = serving_app.FIELD_ORDER[0], serving_app.FIELD_ORDER[1]

# Missing field: drop the first feature. Pydantic must refuse, not default it to 0.
try:
    PredictRequest(**{k: v for k, v in complete.items() if k != first})
except ValidationError as e:
    print(f"Missing '{first}' -> rejected:")
    print(" ", e.errors()[0]["type"], "-", e.errors()[0]["msg"])

# Wrong type: send text where a measurement belongs.
try:
    PredictRequest(**{**complete, second: "not a number"})
except ValidationError as e:
    print(f"\n'{second}' = 'not a number' -> rejected:")
    print(" ", e.errors()[0]["type"], "-", e.errors()[0]["msg"])

print("\nOver HTTP both of these come back as 422 Unprocessable Entity,")
print("before the model is ever called.")


Missing 'mean_radius' -> rejected:
  missing - Field required

'mean_texture' = 'not a number' -> rejected:
  float_parsing - Input should be a valid number, unable to parse string as a number

Over HTTP both of these come back as 422 Unprocessable Entity,
before the model is ever called.


## 💬 Discuss

The cell above rejected a missing field and a non-numeric field with a `422`, *before* the model ran. Now argue the cases the printed output does **not** settle.

1. A client sends all the right numbers in the right format but in the **wrong order** — the texture value in the `mean_radius` field. Our schema accepts it and the model answers confidently. Whose bug is that: the API's, the client's, or the card's? What would you actually add to catch it, knowing every extra check is a check somebody has to maintain forever?
2. An integration team asks you to make missing fields default to the training-set mean instead of returning `422`, "so the dashboard never breaks". Make the strongest case for each side. Which failure would you rather explain to a manager afterwards — a visibly broken dashboard, or a dashboard that quietly displayed predictions built from invented inputs?
3. `/health` reports that the service is up and which model is loaded. Suppose the service is up, the model is the right one, and every prediction is wrong because the upstream system started sending millimetres where the model was trained on centimetres. What would `/health` have to measure to catch that — and at what point does it stop being a health check and become monitoring (Unit 5)?

## Summary

Model serving means wrapping **an artifact somebody already trained** in an HTTP layer so any client can reach it. A FastAPI service has four essential parts:

| Component | What it does |
|---|---|
| Model artifact (`model.joblib` / `model.onnx`) | The trained parameters on disk — your model from AIAT 114 or AIAT 122 |
| `model_card.json` | Feature order, class names, sample input, honest held-out score |
| Generated Pydantic schema | One validated field per feature — built from the card, so it cannot drift |
| API endpoints (`/predict`, `/health`) | Expose the model over HTTP; health reports which model is actually live |

Loading the artifact at import (once per process, not per request) keeps latency low. The `/health` route lets orchestrators know the service is ready — and, because it echoes the card, tells you whether the box is serving a student model or the course fallback.

The same two files travel through the rest of this course: Unit 2 serves them behind Flask and FastAPI, Unit 4 copies them into a Docker image.

## Self-check

Answer from memory first, then verify by re-reading the notebook.

1. **Why does the API build its request schema from `model_card.json` instead of hard-coding field names?** Hint: imagine you retrain with one extra feature and redeploy.
2. **What HTTP status code does FastAPI return when a required field is missing?** Run the validation cell and check the error type — then look up what uvicorn would return to a client.
3. **Why do we load the artifact at module import rather than inside the `/predict` function?** Think about what happens to latency if the model is loaded on every request.
4. **The card stores `feature_names` in order. What goes wrong if a client sends the right values under the wrong field names?** Would the API notice?

## ⚠️ Where this breaks

One process, one artifact loaded at import, a typed schema generated from the card — that is the right default for a single CPU model. It stops being right in specific, nameable places.

- **The schema checks type and presence, never meaning.** Pydantic proved a field is a `float`; it cannot know that `12.4` is a plausible *mean radius*. The Cloudflare outage above was exactly this class of failure: data that was perfectly well-formed and still fatal. If wrong-but-valid input is expensive for you, range and distribution checks belong in the handler (notebook 05) or in monitoring (Unit 5) — not in the type system.
- **In-process calls are not a load test.** Calling `serving_app.predict()` skips the network, the ASGI server and concurrency entirely. It proves the code path is right; it says nothing about behaviour at 200 requests/second. Notebook 03 measures latency, Unit 2 notebook 06 measures throughput.
- **One model per process is a memory decision, not a detail.** Loading at import means every worker holds its own copy. Harmless for this artifact; for a 500 MB model behind 4 Gunicorn workers it is 2 GB of RAM before the first request arrives (Unit 2, notebook 01).
- **The assumption that must hold:** the card sitting next to the artifact actually describes that artifact. Nothing in this notebook verifies it. Re-export `model.joblib` without re-exporting `model_card.json` and the API will validate incoming JSON flawlessly against the schema of a model that no longer exists. Unit 2's golden batch is the cheap check that catches it.
- **The cheaper alternative.** If the only caller is another Python service in the same cluster, an HTTP hop buys you serialization cost and a new failure mode for nothing — import the model instead. If the caller is a nightly job over a million rows, an API is the wrong shape entirely: batch scoring (Unit 2, notebook 06) is faster per row by orders of magnitude.

## 📚 References

1. Sculley, D., Holt, G., Golovin, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS 28. <https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html>
2. Crankshaw, D., Wang, X., Zhou, G., Franklin, M. J., Gonzalez, J. E., & Stoica, I. (2017). *Clipper: A Low-Latency Online Prediction Serving System*. NSDI. <https://arxiv.org/abs/1612.03079>
3. Kreuzberger, D., Kühl, N., & Hirschl, S. (2022). *Machine Learning Operations (MLOps): Overview, Definition, and Architecture*. IEEE Access. <https://arxiv.org/abs/2205.02302>
